# Student Performance Prediction & Academic Risk Analysis
## Exploratory Data Analysis & Early-Warning Feature Isolation

**Student:** Mutyala Pramod Abhiram  
**Roll Number:** 238T1A05B2  
**Institution:** Dhanekula Institute of Engineering & Technology (DIET), Vijayawada  
**Department:** Computer Science and Engineering  
**Dataset:** UCI Machine Learning Repository - Student Performance (Cortez & Silva, 2008)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams.update({"font.size": 11, "figure.autolayout": True})

# Locate raw dataset
data_path = Path("../data/raw/student-mat.csv")
if not data_path.exists():
    data_path = Path("data/raw/student-mat.csv")

df = pd.read_csv(data_path, sep=";")
print(f"Loaded Student Performance dataset: {df.shape[0]} rows, {df.shape[1]} columns.")

### 1. Data Integrity and Missing Value Analysis

In [ ]:
print("Missing values per column:")
print(df.isna().sum().value_counts())
print(f"Total duplicated rows: {df.duplicated().sum()}")

### 2. Summary Statistics of Numerical Indicators and Target (G3)

In [ ]:
df.describe().T

### 3. Final Grade (G3) Distribution & Academic Risk Tiers
In Portuguese secondary education, student achievement is evaluated on a 0 to 20 integer scale:
- **< 10:** High Academic Risk (Failing mark)
- **10 to 13:** Moderate Academic Risk (Satisfactory passing mark)
- **>= 14:** Low Academic Risk (Good to Excellent performance)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sns.histplot(df["G3"], kde=True, bins=21, color="#0d9488", ax=ax)
ax.axvline(10, color="#ef4444", linestyle="--", label="Passing Threshold (10/20)")
ax.axvline(df["G3"].mean(), color="#0284c7", linestyle="-", label=f"Mean ({df['G3'].mean():.2f})")
ax.set_title("Distribution of Final Mathematics Grades (G3)", fontsize=13, fontweight="bold")
ax.set_xlabel("G3 Final Grade (0-20)")
ax.set_ylabel("Student Count")
ax.legend()
plt.show()

### 4. Correlation Analysis Among Numerical Attributes

In [ ]:
num_cols = df.select_dtypes(include=[np.number]).columns
plt.figure(figsize=(12, 10))
sns.heatmap(df[num_cols].corr(), annot=True, fmt=".2f", cmap="coolwarm", square=True)
plt.title("Numerical Correlation Matrix", fontsize=14, fontweight="bold")
plt.show()

### 5. Study Habits, Failures and Absence Impact

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.boxplot(x="studytime", y="G3", data=df, ax=axes[0], palette="crest")
axes[0].set_title("Weekly Study Time vs Final Grade", fontweight="bold")
axes[0].set_xlabel("Weekly Study Time (1: <2h, 2: 2-5h, 3: 5-10h, 4: >10h)")

sns.scatterplot(x="absences", y="G3", hue="failures", data=df, palette="magma", s=70, ax=axes[1])
axes[1].set_title("School Absences & Past Failures vs Final Grade", fontweight="bold")
axes[1].set_xlabel("Number of Absences")
plt.show()

### 6. Early-Warning Feature Isolation Guarantee
To prevent circular grade leakage, `G1`, `G2`, and `G3` are isolated from early-warning modeling.

In [ ]:
X = df.drop(columns=["G1", "G2", "G3"])
y_reg = df["G3"]
print(f"Early-warning feature matrix shape: {X.shape}")
print(f"Target vector shape: {y_reg.shape}")